In [1]:
pip install pyspark

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Pivot Table and Cross Tab Example") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


24/09/14 04:29:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
data = [
    ("John", "Finance", 3000),
    ("Jane", "Marketing", 4000),
    ("John", "Marketing", 2000),
    ("Doe", "Finance", 4000),
    ("John", "Finance", 1000),
    ("Jane", "Marketing", 1000),
    ("Doe", "Marketing", 2000)
]

columns = ["Employee", "Department", "Salary"]
df = spark.createDataFrame(data, columns)
df.show()

+--------+----------+------+
|Employee|Department|Salary|
+--------+----------+------+
|    John|   Finance|  3000|
|    Jane| Marketing|  4000|
|    John| Marketing|  2000|
|     Doe|   Finance|  4000|
|    John|   Finance|  1000|
|    Jane| Marketing|  1000|
|     Doe| Marketing|  2000|
+--------+----------+------+



In [4]:
pivot_df = df.groupBy("Employee").pivot("Department").sum("Salary")
pivot_df.show()

+--------+-------+---------+
|Employee|Finance|Marketing|
+--------+-------+---------+
|     Doe|   4000|     2000|
|    John|   4000|     2000|
|    Jane|   null|     5000|
+--------+-------+---------+



In [5]:
cross_tab_df = df.stat.crosstab("Employee", "Department")
cross_tab_df.show()

+-------------------+-------+---------+
|Employee_Department|Finance|Marketing|
+-------------------+-------+---------+
|               John|      2|        1|
|               Jane|      0|        2|
|                Doe|      1|        1|
+-------------------+-------+---------+



In [6]:
from pyspark.sql import functions as F

advanced_pivot_df = df.groupBy("Employee") \
    .pivot("Department") \
    .agg(
        F.sum("Salary").alias("Total_Salary"),
        F.avg("Salary").alias("Average_Salary"),
        F.max("Salary").alias("Max_Salary")
    )
advanced_pivot_df.show()

+--------+--------------------+----------------------+------------------+----------------------+------------------------+--------------------+
|Employee|Finance_Total_Salary|Finance_Average_Salary|Finance_Max_Salary|Marketing_Total_Salary|Marketing_Average_Salary|Marketing_Max_Salary|
+--------+--------------------+----------------------+------------------+----------------------+------------------------+--------------------+
|     Doe|                4000|                4000.0|              4000|                  2000|                  2000.0|                2000|
|    John|                4000|                2000.0|              3000|                  2000|                  2000.0|                2000|
|    Jane|                null|                  null|              null|                  5000|                  2500.0|                4000|
+--------+--------------------+----------------------+------------------+----------------------+------------------------+--------------------+

In [7]:
departments = df.select("Department").distinct().rdd.flatMap(lambda x: x).collect()
departments = departments[:10]  # Limit to top 10 departments if there are many
pivot_df = df.groupBy("Employee").pivot("Department", departments).sum("Salary")
pivot_df.show()

+--------+-------+---------+
|Employee|Finance|Marketing|
+--------+-------+---------+
|     Doe|   4000|     2000|
|    John|   4000|     2000|
|    Jane|   null|     5000|
+--------+-------+---------+



In [8]:
spark.stop()